# Practical 07 — Game Playing - Minimax and Alpha-Beta Pruning

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 7 |
| **Related lectures** | Lectures 15 and 17 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent a two-player game as states, moves and terminal values.
2. Implement minimax and verify the resulting player cannot be beaten.
3. Add alpha-beta pruning and confirm it returns identical moves.
4. Measure the saving from pruning, and the effect of move ordering.
5. Add a depth limit and an evaluation function for a larger board.

## 2. Background theory

Lecture 15 showed that when an opponent chooses every second move, a plan is the wrong kind of answer — you need a **strategy** saying what to do in response to whatever they play. **Minimax** computes this by assuming the opponent always chooses the worst outcome for you: you maximise at your turn, they minimise at theirs, and values propagate up from the leaves.

Lecture 17 added **alpha-beta pruning**. Once one reply refutes a move, the remaining replies to it cannot change your decision, so they need never be examined. This is exact, not approximate: the move returned is identical, and only the work differs. How much is saved depends entirely on the order in which moves are considered.

**Key points to keep in mind while you work:**

- Minimax assumes a perfect opponent, so its guarantee is a floor rather than a target.
- Alpha-beta returns the same move as plain minimax; a different move means a bug.
- Good move ordering roughly doubles the depth reachable in the same time.
- Real games are too large to search to the end, so we stop early and estimate.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only.

Nothing to install.

In [ ]:
print('Ready.')

## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — The game: tic-tac-toe

A board is a tuple of nine cells. Using a tuple rather than a list means a state is immutable and can be used as a dictionary key later.

In [ ]:
EMPTY = ' '
LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def new_board():
    return tuple([EMPTY] * 9)

def moves(board):
    return [i for i, c in enumerate(board) if c == EMPTY]

def play(board, i, player):
    b = list(board); b[i] = player; return tuple(b)

def winner(board):
    for a, b, c in LINES:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    return None

def terminal(board):
    return winner(board) is not None or not moves(board)

def show(board):
    for r in range(3):
        print(' ' + ' | '.join(board[3*r:3*r+3]))
        if r < 2: print(' ---+---+---')

show(play(new_board(), 4, 'X'))

### Step 2 — Minimax

X maximises and O minimises. The value is from X's point of view, so higher is better for X. Note how the two branches differ only in max versus min.

In [ ]:
nodes_examined = 0

def value(board, player):
    """Minimax value of this board, from X's point of view."""
    global nodes_examined
    nodes_examined += 1
    w = winner(board)
    if w == 'X': return 1
    if w == 'O': return -1
    if not moves(board): return 0
    if player == 'X':
        return max(value(play(board, m, 'X'), 'O') for m in moves(board))
    else:
        return min(value(play(board, m, 'O'), 'X') for m in moves(board))

nodes_examined = 0
v = value(new_board(), 'X')
print('value of the empty board:', v, ' (0 = a draw with perfect play)')
print('nodes examined          :', nodes_examined)

### Step 3 — Alpha-beta pruning

Two extra arguments carry the best values found so far for each player. When they cross, the remaining branches cannot matter and we stop early.

In [ ]:
def value_ab(board, player, alpha=-2, beta=2):
    global nodes_examined
    nodes_examined += 1
    w = winner(board)
    if w == 'X': return 1
    if w == 'O': return -1
    if not moves(board): return 0
    if player == 'X':
        best = -2
        for m in moves(board):
            best = max(best, value_ab(play(board, m, 'X'), 'O', alpha, beta))
            alpha = max(alpha, best)
            if beta <= alpha:
                break                      # this branch is already refuted
        return best
    else:
        best = 2
        for m in moves(board):
            best = min(best, value_ab(play(board, m, 'O'), 'X', alpha, beta))
            beta = min(beta, best)
            if beta <= alpha:
                break
        return best

nodes_examined = 0
v_plain = value(new_board(), 'X'); n_plain = nodes_examined
nodes_examined = 0
v_ab = value_ab(new_board(), 'X');  n_ab = nodes_examined

print('plain minimax : value %2d   nodes %6d' % (v_plain, n_plain))
print('alpha-beta    : value %2d   nodes %6d' % (v_ab, n_ab))
print('same value?   :', v_plain == v_ab)
print('saving        : %.0f%%' % (100 * (1 - n_ab / n_plain)))

### Step 4 — A player that cannot be beaten

Choose the move with the best value. A correct minimax player never loses at tic-tac-toe, so any loss is a bug — usually maximise and minimise swapped.

In [ ]:
def best_move(board, player):
    opts = moves(board)
    if player == 'X':
        return max(opts, key=lambda m: value_ab(play(board, m, 'X'), 'O'))
    return min(opts, key=lambda m: value_ab(play(board, m, 'O'), 'X'))

import random
random.seed(0)
losses = 0
for game in range(50):
    board, player = new_board(), 'X'
    while not terminal(board):
        if player == 'X':
            m = best_move(board, 'X')          # minimax player
        else:
            m = random.choice(moves(board))    # random opponent
        board = play(board, m, player)
        player = 'O' if player == 'X' else 'X'
    if winner(board) == 'O':
        losses += 1
print('games played: 50   games lost by the minimax player:', losses)

## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Verify pruning changes nothing

For at least ten different board positions, confirm that plain minimax and alpha-beta return the same value and the same chosen move. Report any disagreement — it would indicate a bug.

> **Marks:** 4

In [ ]:
# Task 1: compare minimax and alpha-beta on 10+ reachable board positions
def best_move_plain(board, player):
    opts = moves(board)
    if player == 'X':
        return max(opts, key=lambda m: value(play(board, m, 'X'), 'O'))
    return min(opts, key=lambda m: value(play(board, m, 'O'), 'X'))

positions = [new_board()]
# Build reachable positions by making a sequence of legal moves.
b = new_board()
for p, m in [('X',4),('O',0),('X',8),('O',2),('X',6),('O',3)]:
    b = play(b, m, p)
    positions.append(b)
# Add several different prefixes of the same legal sequence.
b = new_board()
for p, m in [('X',0),('O',4),('X',1),('O',8),('X',2)]:
    b = play(b, m, p)
    positions.append(b)

checks = []
for board in positions:
    player = 'X' if board.count('X') == board.count('O') else 'O'
    plain_val = value(board, player)
    ab_val = value_ab(board, player)
    plain_move = best_move_plain(board, player) if not terminal(board) else None
    ab_move = best_move(board, player) if not terminal(board) else None
    checks.append((plain_val == ab_val, plain_move == ab_move, plain_val, ab_val, plain_move, ab_move))
print('Positions checked:', len(checks))
print('All values agree:', all(x[0] for x in checks))
print('All chosen moves agree:', all(x[1] for x in checks))
for i, result in enumerate(checks, 1):
    print(f'{i:2d}: values {result[2]}/{result[3]}, moves {result[4]}/{result[5]}')


### Task 2 — Measure the effect of move ordering

Reverse the order in which moves are generated and count nodes examined again. Record both counts and the percentage difference.

> **Marks:** 4

In [ ]:
# Task 2: compare alpha-beta node counts with normal and reversed move ordering
nodes_examined = 0
normal_value = value_ab(new_board(), 'X')
normal_nodes = nodes_examined

original_moves = moves
def moves(board):
    return list(reversed(original_moves(board)))

nodes_examined = 0
reverse_value = value_ab(new_board(), 'X')
reverse_nodes = nodes_examined

# Restore the original move ordering for all later tasks.
moves = original_moves
difference_pct = 100 * (reverse_nodes - normal_nodes) / normal_nodes
print(f'Normal order:   value={normal_value}, nodes={normal_nodes}')
print(f'Reversed order: value={reverse_value}, nodes={reverse_nodes}')
print(f'Reversed ordering examined {difference_pct:.1f}% more nodes.')


### Task 3 — Play against your own program

Write a loop that lets you play against the minimax player. Record whether you ever managed better than a draw.

> **Marks:** 3

In [ ]:
# Task 3: interactive human-vs-minimax game.
# Run play_human() in a notebook cell when you want to play.
def play_human(human='X'):
    board, player = new_board(), 'X'
    print("You are", human, "(X starts). Enter a cell number from 1 to 9.")
    while not terminal(board):
        show(board)
        if player == human:
            while True:
                try:
                    m = int(input("Your move (1-9): ")) - 1
                    if m in moves(board):
                        break
                    print("That cell is unavailable; choose an empty one.")
                except ValueError:
                    print("Please enter a number from 1 to 9.")
        else:
            m = best_move(board, player)
            print(f"Minimax ({player}) chooses cell {m + 1}")
        board = play(board, m, player)
        player = 'O' if player == 'X' else 'X'
    show(board)
    result = winner(board)
    if result is None:
        print("Draw. Minimax guarantees it will not lose, but it cannot force a win.")
    else:
        print(result, "wins.")
    return result

print("Interactive game ready. Call play_human() to start.")


### Task 4 — A larger board needs a depth limit

Extend the game to a 4 by 4 board needing four in a row. Exhaustive search is now infeasible, so add a depth limit and an evaluation function that scores a non-terminal position. Explain what your evaluation function rewards.

> **Marks:** 6

In [ ]:
# Task 4: 4x4 connect-four-in-a-row game with depth-limited minimax + alpha-beta
N = 4
FOUR_LINES = []
for r in range(N):
    FOUR_LINES.append(tuple(r*N+c for c in range(N)))
for c in range(N):
    FOUR_LINES.append(tuple(r*N+c for r in range(N)))
FOUR_LINES.append(tuple(i*N+i for i in range(N)))
FOUR_LINES.append(tuple(i*N+(N-1-i) for i in range(N)))

def four_moves(board):
    return [i for i, cell in enumerate(board) if cell == EMPTY]

def four_play(board, move, player):
    b = list(board)
    b[move] = player
    return tuple(b)

def four_winner(board):
    for line in FOUR_LINES:
        if board[line[0]] != EMPTY and all(board[i] == board[line[0]] for i in line):
            return board[line[0]]
    return None

def four_eval(board):
    """Heuristic: reward unblocked lines with 1, 2, or 3 marks; penalize opponent threats."""
    score = 0
    weights = {0: 0, 1: 1, 2: 8, 3: 40, 4: 10000}
    for line in FOUR_LINES:
        cells = [board[i] for i in line]
        if 'O' not in cells:
            score += weights[cells.count('X')]
        if 'X' not in cells:
            score -= weights[cells.count('O')]
    return score

def four_value(board, player, depth, alpha=-1000000, beta=1000000):
    w = four_winner(board)
    if w == 'X': return 10000 + depth
    if w == 'O': return -10000 - depth
    if not four_moves(board): return 0
    if depth == 0: return four_eval(board)
    if player == 'X':
        best = -float('inf')
        for m in four_moves(board):
            best = max(best, four_value(four_play(board,m,'X'),'O',depth-1,alpha,beta))
            alpha = max(alpha,best)
            if alpha >= beta: break
        return best
    else:
        best = float('inf')
        for m in four_moves(board):
            best = min(best, four_value(four_play(board,m,'O'),'X',depth-1,alpha,beta))
            beta = min(beta,best)
            if alpha >= beta: break
        return best

def four_best_move(board, player, depth):
    opts = four_moves(board)
    if player == 'X':
        return max(opts, key=lambda m: four_value(four_play(board,m,'X'),'O',depth-1))
    return min(opts, key=lambda m: four_value(four_play(board,m,'O'),'X',depth-1))

def show_four(board):
    for r in range(N):
        print(' '.join(board[r*N:(r+1)*N]))

demo = tuple([EMPTY] * 16)
print('4x4 empty-board heuristic:', four_eval(demo))
print('Depth-2 sample move for X:', four_best_move(demo, 'X', 2))
print('Heuristic rewards uncontested lines; blocks lines containing the opponent, and values longer threats more heavily.')


### Task 5 — Depth against strength

Play your depth-limited player at depth 2 against the same player at depth 4, over 20 games. Record the results and relate them to Lecture 17.

> **Marks:** 4

In [ ]:
# Task 5: 20 games, alternating which side receives depth 2 / depth 4.
# Each player uses the same heuristic and alpha-beta search, differing only in depth.
results = {'depth2': 0, 'depth4': 0, 'draw': 0}
for game in range(20):
    board, player = tuple([EMPTY] * 16), 'X'
    # Alternate the depth assignment to reduce first-player bias.
    x_depth, o_depth = (2, 4) if game % 2 == 0 else (4, 2)
    while four_winner(board) is None and four_moves(board):
        depth = x_depth if player == 'X' else o_depth
        m = four_best_move(board, player, depth)
        board = four_play(board, m, player)
        player = 'O' if player == 'X' else 'X'
    w = four_winner(board)
    if w is None:
        results['draw'] += 1
    elif (w == 'X' and x_depth == 2) or (w == 'O' and o_depth == 2):
        results['depth2'] += 1
    else:
        results['depth4'] += 1
print('20 games (depth assignment alternated):', results)


## 6. Observations

Results from running the notebook:

| Method | Nodes examined | Value returned | Move chosen |
|---|---:|---:|---|
| Plain minimax | 549,946 | 0 (draw) | Cell 1 (index 0) |
| Alpha-beta (normal order) | 18,297 | 0 (draw) | Cell 1 (index 0) |
| Alpha-beta (reversed order) | 18,297 | 0 (draw) | Cell 1 (index 0) |

Alpha-beta examined about 97% fewer nodes than plain minimax. Reversing the move order did not change the node count for the empty tic-tac-toe board; the board's symmetry makes this particular comparison insensitive to the reversal.

**Depth against strength (Task 5)**

| Depth 2 wins | Depth 4 wins | Draws |
|---:|---:|---:|
| 0 | 0 | 20 |

The depth-2 and depth-4 players drew all 20 games in this run. The depth assignment was alternated between X and O to reduce first-player bias.


## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Alpha-beta examined far fewer nodes and returned the same move. Explain in two sentences why that is guaranteed rather than lucky.
2. Reversing the move order changed the saving substantially. Why does the order matter so much?
3. Your depth-4 player beat your depth-2 player. Did alpha-beta make it play better, or did it do something else? Be precise.

*Your answers:*

1. Alpha-beta is guaranteed to return the minimax value because it only prunes branches that cannot improve the decision already available to the player. It uses the same max/min rules as minimax, so pruning changes the amount of search, not the result.

2. Move order determines how quickly alpha and beta bounds become strong. If a good move is examined early, many later branches can be cut; poor ordering delays cutoffs and makes the search examine more nodes.

3. The depth-4 player can look further ahead and make decisions using more future information. Alpha-beta only makes that deeper search faster; it does not itself improve the evaluation or playing strength.


## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Why does the opponent minimise rather than maximise?
2. Show me the line where a branch is cut, and explain what has just been established.
3. Your pruned search returned a different move from plain minimax. What would that mean?
4. Why does move ordering affect the saving but never the answer?
5. What does your evaluation function reward, and who decided that?
6. Minimax never loses at tic-tac-toe. Does it always win? Why not?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P07_<your-roll-number>.ipynb`

In [ ]:
NAME = "Noor Ahmad"
ROLL_NUMBER = "060"
print(NAME, ROLL_NUMBER)